# Sprint 4 — Model Research Notes (Part 2)

Continuation of `04_model_research_part1.ipynb` — same format (paper
summary + a separate VisionLab evaluation section).

## Reading Note 4: Very Deep Convolutional Networks for Large-Scale Image Recognition (VGG)

**Source:** Simonyan, K., & Zisserman, A. (2015).
*Very Deep Convolutional Networks for Large-Scale Image Recognition.* ICLR 2015.
(ILSVRC 2014 localization winner)

### 1. What does the paper argue?

The paper isolates a single variable: depth. Everything else (filter size,
layer arrangement) is held fixed while layer count is scaled up
systematically. AlexNet and the other strong nets of the time (Zeiler &
Fergus, OverFeat) used large receptive fields in the first layer (11×11
stride 4, 7×7 stride 2). VGG instead uses only 3×3 filters throughout the
whole network — the smallest size that can still capture
left/right/up/down/center.

**Why only 3×3?** The paper's most elegant argument: two stacked 3×3 conv
layers cover the same effective receptive field as one 5×5 layer; three
stacked 3×3 layers match one 7×7. So why prefer the small-filter stack over
one big filter? Two reasons: (1) **more non-linearity** — three 3×3 layers
means three separate ReLUs instead of one, making the decision function
more discriminative; (2) **fewer parameters** — a 3×(3²C²) = 27C² stack
uses 81% fewer weights than a single 7²C² = 49C² layer for `C` channels,
acting as an implicit regularizer.

**Architecture family (A–E):** keeping the same skeleton (conv stack →
maxpool → ... → 3 FC layers → softmax), the paper compares 6 configurations
going from 11 layers (A) to 19 layers (E — today's "VGG-19"), doubling
channel width after each maxpool (64→128→256→512).

### Findings

- LRN (Local Response Normalization) gives no benefit, only extra
  memory/compute cost — dropped from every configuration except A-LRN.
- Error decreases systematically with depth (11 → 19 layers), but the gain
  saturates at 19 layers; the authors note deeper models may pay off on
  larger datasets.
- Filter choice matters even at equal depth: configuration C (some 3×3s
  replaced by 1×1) is worse than D (all 3×3) — extra non-linearity (1×1)
  helps, but filters that capture spatial context (3×3) matter more.
- Direct comparison: a shallow variant of net B with its 3×3 pairs replaced
  by a single 5×5 is 7% worse (top-1) than B — "deep + small filter" beats
  "shallow + large filter" directly, not just in theory.
- Scale jittering (multi-scale training) helps clearly: sampling the
  training image's shortest side from `[256, 512]` instead of a fixed `S`
  improves results even when testing at a single scale.
- Weight init: the deepest models weren't trained from random init directly
  — the shallow model A is trained first, and its first 4 conv layers +
  last 3 FC layers seed the deeper models. The authors later note this
  turned out to be unnecessary once Glorot/Bengio init was available.
- Test-time strategy matters too: converting FC layers to convolutional
  ones (fully-convolutional) for dense evaluation over the whole image,
  combined with multi-scale and horizontal-flip averaging, clearly beats
  single-crop evaluation; combining dense with multi-crop is better than
  either alone (complementary boundary/padding effects).
- Generalization (Appendix B): VGG features trained on ILSVRC, used
  *without* fine-tuning — just the last layer removed and a linear SVM
  trained on top — beat the era's state-of-the-art on VOC-2007/2012 and
  Caltech-101/256. Early, strong evidence that a good backbone also works
  as a general-purpose feature extractor.

## 2. Evaluation for VisionLab

This paper feeds directly into both the Model Zoo (architecture) and
Training Engine (augmentation) sprints.

**Concrete decisions this supports:**

- **"Small filter + depth beats large filter + shallowness"** would be a
  direct reference if VGG is added to the registry as a backbone option —
  but more importantly, this principle is already the design philosophy
  behind modern backbones (ResNet included). Worth a short line in the
  Model Zoo docs answering "why did 3×3 convolutions become the standard."
- **The scale jittering finding directly supports and grounds our
  parametric augmentation preset decision (`none`/`light`/`medium`/`heavy`)
  from Sprint 3.** What VGG does — sampling the training image's shortest
  side from a range (`[256, 512]`) instead of a fixed value — is the scale
  component of a `RandomResizedCrop`-style augmentation. Including this
  kind of scale jitter in our `medium`/`heavy` presets can be prioritized
  since it's a literature-proven gain, not a guess.
- **Test-time augmentation (dense + multi-crop + horizontal-flip
  averaging) is a directly applicable technique for the Inference sprint:**
  converting the model's FC layers to convolutional form to process the
  full image (no cropping) plus averaging predictions across multiple
  scales can improve accuracy with no extra training. This could be added
  to config as an eval-time option in the Benchmark sprint (e.g.
  `tta: {enabled: true, scales: [256, 384, 512], hflip: true}`).
- **Getting SOTA on other datasets via feature extraction + a linear SVM,
  with no fine-tuning (Appendix B), directly supports the "frozen backbone
  + new classifier" scenario in our Model Zoo** — especially if our
  mushroom/flower datasets end up small, taking a baseline with a fully
  frozen backbone + a simple classifier (linear/MLP) first offers a
  low-cost starting point alongside the freeze-vs-fine-tune decision from
  our Yosinski note.
- **The shallow-to-deep weight initialization strategy is an interesting
  historical detail, but a problem already superseded by Glorot/Kaiming
  init** — defaulting to modern init schemes (Kaiming He init, also
  referenced in our ResNet note) in the Model Zoo lines up with what the
  paper itself later admits.
- **VGG's very high parameter count (134–144M) is a cost worth flagging:**
  as noted in our ResNet note, even ResNet-152 uses fewer FLOPs than
  VGG-16/19. If VGG is offered as a Model Zoo option, documenting it as
  "high memory/parameter cost, but a classic backbone with proven
  generalization" sets the right expectation for whoever revisits the
  project later (including future us).

**Overall takeaway:** VGG is the clearest evidence for the "simple but
deep" philosophy and the origin of the 3×3-convolution standard in modern
backbone design. Beyond being a direct backbone candidate for the Model
Zoo, its real value here is that the scale jittering and test-time
augmentation findings are two concrete, experimentally-proven techniques
that transfer directly into Sprint 3's augmentation design and the future
Inference sprint.

## Reading Note 5: EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks

**Source:** Tan, M., & Le, Q. V. (2019).
*EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML 2019.

### 1. What does the paper argue?

VGG isolated depth; ResNet chased the degradation problem. This paper asks
a different question: when scaling up a ConvNet (given more compute), which
dimension should grow — depth, width, or resolution? The answer is more
interesting than most prior work assumed: all three together, in balance.

**Three scaling dimensions:** depth `d` (more layers — richer features, but
harder to train and diminishing returns past a point, e.g. ResNet-1000 ≈
ResNet-101 in accuracy), width `w` (more channels per layer — captures
fine-grained features but wide+shallow nets struggle with high-level
features), and resolution `r` (larger input images — more fine detail, but
diminishing returns too).

**Observation 1:** scaling any single dimension alone improves accuracy,
but the gain saturates quickly (Fig. 3) — past a point, extra compute is
nearly wasted.

**Observation 2 (the actual finding):** these three dimensions aren't
independent. Higher resolution needs more depth to grow the receptive
field accordingly, and more width to capture finer-grained patterns at that
resolution. Scaling width alone saturates fast; scaling depth+resolution
together at the same FLOP budget does much better (Fig. 4).

**Compound scaling:** a single compound coefficient `φ` scales all three
dimensions together at fixed ratios:

```
depth:      d = α^φ
width:      w = β^φ
resolution: r = γ^φ
constraint: α · β² · γ² ≈ 2
```

`α, β, γ` are found via a small grid search (for EfficientNet: α=1.2,
β=1.1, γ=1.15); `φ` is the one user-facing "how much more compute" knob —
increasing it by 1 roughly doubles FLOPs, distributed proportionally across
the three dimensions. The method applies both to existing architectures
(ResNet, MobileNet — Table 3) and to a new NAS-discovered baseline,
EfficientNet-B0, scaled up into the B1–B7 family.

**EfficientNet-B0:** found via neural architecture search optimizing
accuracy and FLOPs jointly; built from MBConv blocks (mobile inverted
bottleneck, from MobileNetV2) plus squeeze-and-excitation. B1–B7 share the
same α/β/γ ratios — only `φ` changes.

### Findings

- EfficientNet-B7 reaches 84.3% top-1 (ImageNet) — matching the previous
  state-of-the-art (GPipe) with 8.4× fewer parameters and 6.1× faster
  inference.
- EfficientNet-B4 vs. ResNet-50 at similar FLOPs: 76.3% → 83.0% (+6.7
  points) — a clearly better architecture/scaling combination at the same
  compute budget.
- Compound scaling systematically beats single-dimension scaling (Fig. 8):
  from the same baseline, balancing depth+width+resolution together gains
  up to 2.5% extra accuracy over scaling just one dimension at the same
  FLOP budget.
- Class Activation Map analysis (Fig. 7): compound-scaled models focus on
  more relevant and complete object regions than single-dimension-scaled
  ones — a qualitative difference in representation quality, not just a
  number.
- **Transfer learning (Table 5) — directly relevant for VisionLab:**
  ImageNet-pretrained EfficientNet tested on 8 transfer datasets (CIFAR-10/100,
  Birdsnap, Stanford Cars, **Flowers**, FGVC Aircraft, Oxford-IIIT Pets,
  Food-101). On Flowers — the exact same Oxford Flowers dataset used in
  VisionLab — EfficientNet-B7 reaches **98.8% accuracy**.

## 3. Evaluation for VisionLab

This paper ties together the previous three notes (Yosinski, ImageNet,
ResNet, VGG) and intersects directly with our second dataset (Flowers).

**Concrete decisions this supports:**

- **A direct numerical reference point on the Flowers dataset:**
  EfficientNet-B7 reaches 98.8% accuracy (Table 5). If EfficientNet becomes
  one of the Benchmark sprint's backbone options, our own results can be
  compared against this number directly — a concrete target for the
  literature comparison.
- **A direct template for adding a B0–B7-style "family" to the registry:**
  compound scaling is a concrete example of how to define a model family
  that scales from one baseline architecture (taking a `num_classes`
  parameter) to B0–B7 via a single `phi` or `model_size` parameter — unlike
  ResNet's "each depth level is its own class" approach, here one
  architecture + one scaling function is enough.
- **The depth/width/resolution balance principle carries a concrete warning
  for our config design:** Sprint 3 likely fixes a single input resolution
  (e.g. 224×224). This paper shows that if we ever move to a bigger/deeper
  backbone, image resolution needs to scale proportionally too — using a
  deeper model with the same low resolution leaves most of the possible
  gain on the table. Worth noting that `input_resolution` in
  `mushroom.yaml`/`flower.yaml` is a parameter that should be considered
  jointly with model depth, not independently.
- **Mobile inverted bottleneck (MBConv) + squeeze-and-excitation** adds a
  third alternative to the bottleneck-block discussion from our ResNet
  note. If compute/memory budget matters for the Model Zoo (e.g. a
  low-resource deployment target), the EfficientNet family offers
  similar-or-better accuracy at far fewer FLOPs than ResNet — a relevant
  selection criterion for the Deployment sprint.
- **Dropout rate scaled up with model size** (0.2 for B0, 0.5 for B7) is an
  interesting practical detail: "bigger model = needs more regularization"
  implies that if we ever use a larger EfficientNet variant (B4+) on our
  smaller mushroom dataset, dropout/augmentation strength should be scaled
  up proportionally too.
- **AutoAugment and stochastic depth usage** gives concrete examples of
  what could sit at the `heavy` end of Sprint 3's augmentation presets
  (`none`/`light`/`medium`/`heavy`) — AutoAugment in particular, as a
  dataset-specific learned augmentation policy, is a useful reference for
  a future "auto" preset idea.

**Overall takeaway:** this paper adds a third strong candidate to the
"which backbone" question (alongside ResNet and VGG), but more importantly
offers a general framework for "how do I scale up an architecture." The
most concrete takeaway for us is a directly comparable literature result on
our own Flowers dataset (98.8%), plus a reminder that resolution/depth/width
need to be considered together in our config design, not independently.

## Reading Note 6: A ConvNet for the 2020s (ConvNeXt)

**Source:** Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T., & Xie, S. (2022).
*A ConvNet for the 2020s.* CVPR 2022. (Facebook AI Research)

### 1. What does the paper argue?

Different context from the previous four notes: once Vision Transformers
(ViT) overtook ConvNets on ImageNet classification in 2020, the field
questioned whether convolution itself was becoming obsolete. Hierarchical
Transformers like Swin (using local-window attention) brought back some
ConvNet design principles (multi-stage, sliding windows) and succeeded as
general-purpose vision backbones too — but that success was always
credited to something intrinsic to Transformers, not to convolution itself.

This paper asks: if we "modernize" a standard ResNet step by step, toward
the design choices a Swin Transformer makes, can a pure ConvNet compete
with Transformers? The answer: yes, comfortably — the resulting family is
ConvNeXt.

**The modernization journey (Fig. 2):** starting from ResNet-50, with no
attention module ever added, the authors apply changes one at a time and
measure ImageNet accuracy at each step:

1. **Updated training recipe** (76.1% → 78.8%): just applying a "modern"
   training procedure — AdamW, 300 epochs, Mixup/CutMix/RandAugment/Random
   Erasing, stochastic depth, label smoothing — with *zero* architecture
   change, already gains 2.7 points. A large part of the ConvNet-vs-Transformer
   gap turns out to come from training procedure, not architecture.
2. **Macro design:** stage block ratio (3,4,6,3) → (3,3,9,3) (mirroring
   Swin-T), and replacing ResNet's 7×7+maxpool stem with a ViT-style 4×4
   stride-4 "patchify" layer (79.5%).
3. **ResNeXt-ify:** depthwise convolution + wider channels (80.5%).
4. **Inverted bottleneck** (from MobileNetV2 — hidden dimension wider than
   input) (80.6%).
5. **Large kernel size:** moving the depthwise conv up and growing the
   kernel from 3×3 to 7×7 (80.6%) — gains saturate past 7×7.
6. **Micro design:** ReLU→GELU, fewer activations per block (81.3%), fewer
   normalization layers (81.4%), BatchNorm→LayerNorm (81.5%), and separate
   downsampling layers between stages (82.0% — now beating Swin-T's 81.3%).

**Result:** ConvNeXt-T reaches 82.0% vs. Swin-T's 81.3% — with no attention
or self-attention module at all, just standard convolution layers
rearranged.

**Key observation:** the authors point out that none of the design choices
credited to Transformers' superiority (patchify stem, inverted bottleneck,
large kernels, fewer norm/activation layers, LayerNorm, separate
downsampling) were actually new — each had been discovered separately in
the ConvNet literature before, just never combined.

### Findings

- The ConvNeXt family (T/S/B/L/XL) scales via channel count and block
  count, similar to ResNet and Swin.
- With ImageNet-22K pretraining, ConvNeXt-XL reaches 87.8% top-1 — on par
  with or better than Swin, usually with higher inference speed.
- Matches or beats Swin on COCO object detection and ADE20K semantic
  segmentation too — the advantage carries over to downstream tasks, not
  just classification.
- On an A100 GPU, ConvNeXt gets 34–49% higher inference throughput than
  Swin (Table 12) — a hardware-level advantage of being fully convolutional.
- Strong robustness results (ImageNet-A/R/Sketch/C) for ConvNeXt-XL, with
  no dedicated robustness module.
- Isotropic ConvNeXt variants (no downsampling, ViT-style fixed resolution)
  match ViT at equal parameter/FLOP budgets — the block design itself is
  competitive even outside a hierarchical architecture.

## 4. Evaluation for VisionLab

This paper adds a fourth and most recent backbone candidate to the Model
Zoo sprint, and also works as a "best practices" summary synthesizing the
architecture discussions from our previous three notes (ResNet, VGG,
EfficientNet).

**Concrete decisions this supports:**

- **The single most critical and directly applicable finding: the training
  recipe alone gains 2.7 points, with zero architecture change.** This is a
  strong priority signal for our Training Engine sprint: a "modern training
  recipe" preset — AdamW optimizer, RandAugment/Mixup/CutMix, stochastic
  depth, label smoothing — is probably the highest return-on-effort
  improvement available regardless of which backbone (ResNet, EfficientNet,
  ConvNeXt) we pick. A config distinction like `training_recipe: {classic,
  modern}` could directly capture this finding.
- **A concrete update to our registry's block-design discussion** (the
  bottleneck distinction from our ResNet note): the ConvNeXt block
  (depthwise 7×7 conv → LayerNorm → 1×1 expand → GELU → 1×1 project) is
  different from ResNet's bottleneck but can similarly be modeled as its
  own `block_type` in the registry. For fine-grained classification tasks
  like mushroom/flower, the large-kernel depthwise convolution (7×7)
  capturing wider contextual information could be useful where
  texture/shape discrimination matters.
- **The "separate downsampling layers + normalization" finding** (step 6,
  one of the single biggest contributors — 81.5% → 82.0%) isn't directly
  related to our Sprint 3 `ImageClassificationDataset`/pipeline
  architecture, but suggests adding "a separate normalization + downsampling
  layer at every stage transition" as a design rule for the Model Zoo's
  architecture templates.
- **The isotropic ConvNeXt finding** (a downsampling-free variant running
  at constant resolution) is worth keeping in mind as an alternative
  architecture if a future segmentation or Grad-CAM/Error Analysis sprint
  needs to preserve spatial resolution (e.g. an activation-map extraction
  scenario).
- **The inference speed/efficiency comparison (Table 12) reinforces the
  message from our EfficientNet note that FLOPs alone aren't a sufficient
  efficiency metric:** ConvNeXt and Swin sit at similar FLOPs but differ a
  lot in real hardware inference speed (low-level details like
  channel-last memory layout come into play). This supports looking at
  actual measured hardware inference speed, not just FLOPs/parameter count,
  in the Deployment sprint.
- **Robustness evaluation (ImageNet-A/R/Sketch/C)** suggests a dimension
  worth adding to the Error Analysis sprint: for our mushroom/flower
  datasets too, looking at robustness to out-of-distribution samples
  (different lighting, angle, background) — not just standard test-set
  accuracy — can give extra signal on how reliable the model actually is in
  the real world.

**Overall takeaway:** beyond adding a strong, current fourth candidate
(ConvNeXt) to the "which backbone" question, this paper's real message is
that training procedure matters as much as architecture, and that many
small design decisions scattered across the ConvNet literature over the
last decade can add up to a large gain when combined. Keeping VisionLab's
Model Zoo / Training Engine separation clean (architecture in the registry,
training recipe in config) lines up exactly with this paper's main finding.

## Summary — Model Research Wrap-Up & Trial Order

Six papers reviewed across Part 1 and Part 2 (Yosinski et al. on transfer,
Deng et al. on ImageNet, He et al. on ResNet, Simonyan & Zisserman on VGG,
Tan & Le on EfficientNet, Liu et al. on ConvNeXt) converge on a concrete
decision for which backbone to try first in the Model Zoo, and in what
order.

**Decision**

**First model to try: ResNet-50.**
Lowest-risk option — strong literature backing, performs well even with a
classic training recipe, and a simple, well-documented reference point for
validating the registry/factory pattern design. It makes sense to confirm
the Model Zoo architecture (checkpoint saving, label mapping, registry)
works correctly with the least "surprising" model first.

**Second: EfficientNet-B3.**
Once the pipeline is validated with ResNet-50, this is valuable for the
Benchmark sprint because of a direct literature reference on the Flowers
dataset (98.8% with EfficientNet-B7 — B3 is smaller but the same family).

**Defer ConvNeXt-T for now.**
It promises the best raw accuracy, but its gain depends heavily on a
modern training recipe (AdamW + RandAugment + Mixup/CutMix + stochastic
depth). Trying ConvNeXt before that recipe is in place in the Training
Engine sprint would make it hard to tell whether the architecture or the
missing training recipe is responsible for any result.

**No need to add VGG to the Model Zoo.**
High parameter cost, and current alternatives are better on every axis —
keep it as a historical/educational reference only, not a registry option.